# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [10]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Push Prices Handler loaded at 2026-04-08 14:21:56 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-08


In [11]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
    GROUP BY ALL

'''
df_prices = query_snowflake(PRICES_QUERY)
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Market Data Module loaded at 2026-04-08 14:21:57 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
get_market_signals() function de

In [12]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.10)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 230639 rows (product x cohort)
  With market data: 42364
  With WAC: 75560
  With target margin: 230639


In [22]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(7630,700,'market_50'),(7630,701,'market_50'),(151,701,'market_50'),(990,701,'market_50'),(6935,700,'market_25'),(3,700,'market_50'),(7885,700,'market_50'),(434,704,'market_50'),(305,701,'market_50'),(990,700,'market_50'),(10993,701,'market_50'),(83,700,'market_50'),(1492,700,'market_50'),(248,701,'market_50'),(205,700,'market_50'),(3,701,'market_50'),(2328,700,'market_50'),(141,701,'market_50'),(147,700,'market_50'),(589,701,'market_50'),(856,700,'market_50'),(439,700,'market_50'),(147,701,'market_50'),(11404,704,'market_50'),(8672,701,'market_50'),(25899,701,'market_50'),(2328,1123,'market_50'),(143,700,'market_50'),(362,701,'market_50'),(143,701,'market_50'),(144,700,'market_50'),(7630,1126,'market_50'),(2328,701,'market_50'),(8915,701,'market_50'),(24257,700,'market_50'),(59,700,'market_50'),(33,700,'market_50'),(144,701,'market_50'),(130,704,'market_50'),(305,1124,'market_50'),(146,700,'market_50'),(7887,700,'market_50')

]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)]
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.10)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 41 OK across 31 products × cohorts, 1 errors/skipped

Errors/Skipped:
 product_id  cohort_id              sku    action         status
      11404        704 سندة سكر - 1 كجم market_50 NO MARKET DATA


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,7630,700,فيورى جولد - 400 مل,فيوري,market_50,174.693518,233.00,231.495,231.50,0.245384,OK
1,7630,701,فيورى جولد - 400 مل,فيوري,market_50,174.693518,230.00,232.990,233.00,0.250242,OK
2,151,701,لبن بخيره - 1 لتر,بخيره,market_50,431.060743,450.00,457.000,457.00,0.056760,OK
3,990,701,لبن المراعى كامل دسم بلاستيك - 1 لتر,المراعي البان,market_50,547.070438,590.00,585.000,585.00,0.064837,OK
4,6935,700,كوكاكولا اكشن - 300 مل,كوكا كولا,market_25,116.847788,120.00,114.000,114.00,-0.024981,OK
5,3,700,ارز حبوبة رفيع - 1 كجم,حبوبة,market_50,254.494426,277.00,286.000,286.00,0.110159,OK
6,7885,700,شاى العروسة -. 40 جم,العروسة,market_50,202.910670,212.00,209.990,210.00,0.033759,OK
7,434,704,بيبسى كانز جيب - 240 مل,بيبسي,market_50,289.382923,308.25,303.250,303.25,0.045728,OK
8,305,701,عصير بيتى مانجو - 235 مل,بيتي عصاير,market_50,185.534048,210.00,199.000,199.00,0.067668,OK
9,990,700,لبن المراعى كامل دسم بلاستيك - 1 لتر,المراعي البان,market_50,547.070438,585.00,582.500,582.50,0.060823,OK


In [23]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results

,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,7630,700,فيورى جولد - 400 مل,فيوري,market_50,174.693518,233.00,231.495,231.50,0.245384,OK
3,990,701,لبن المراعى كامل دسم بلاستيك - 1 لتر,المراعي البان,market_50,547.070438,590.00,585.000,585.00,0.064837,OK
4,6935,700,كوكاكولا اكشن - 300 مل,كوكا كولا,market_25,116.847788,120.00,114.000,114.00,-0.024981,OK
6,7885,700,شاى العروسة -. 40 جم,العروسة,market_50,202.910670,212.00,209.990,210.00,0.033759,OK
7,434,704,بيبسى كانز جيب - 240 مل,بيبسي,market_50,289.382923,308.25,303.250,303.25,0.045728,OK
8,305,701,عصير بيتى مانجو - 235 مل,بيتي عصاير,market_50,185.534048,210.00,199.000,199.00,0.067668,OK
9,990,700,لبن المراعى كامل دسم بلاستيك - 1 لتر,المراعي البان,market_50,547.070438,585.00,582.500,582.50,0.060823,OK
11,83,700,جبنة عبور لاند فيتا بيضاء - 250 جم,عبور لاند,market_50,531.514593,585.00,574.000,574.00,0.074016,OK
13,248,701,لبن جهينة كامل الدسم - 1 لتر,جهينة ألبان,market_50,533.298035,580.50,551.460,551.50,0.033004,OK
14,205,700,لبن جهينة مكس شوكولاتة - 200 مل,جهينة ألبان,market_50,267.806773,288.00,287.000,287.00,0.066875,OK


In [24]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 21 products × 5 cohorts = 34 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 34
Price changes to push: 34
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 34

🔗 Mirrored prices to 3 main/general cohorts (+35 rows)
   Cohort 695 ← 700: 17 rows
   Cohort 61 ← 700: 17 rows
   Cohort 698 ← 704: 1 rows

📋 Prepared 63 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (17 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 195.89it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (17 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 190.10it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (1 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 237.07it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 63
Total failed: 0
  Cleanup: removed 16 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 34, 'price_changes': 34, 'pushed': 63, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-08 14:32:46', 'mode': 'live', 'skipped_cohorts': []}
